# NB01 — TA-Pfam Extraction

**Requires BERDL JupyterHub** (or an active `spark_connect_remote` proxy chain locally).

Extracts every gene cluster in `kbase_ke_pangenome` that carries at least one Type II toxin-antitoxin (TA) Pfam signature, joins core/accessory status and species assignment, and persists tidy TSVs for downstream lifestyle analysis.

## Panel

`data/ta_families_seed.tsv` lists Pfam NAMES (not PF accessions) — the `PFAMs` column of `eggnog_mapper_annotations` stores comma-delimited names like `RelE`, `HipA_C`, `MazE_antitoxin`. Panel audit in cell 6 reports zero-hit names for revision.

## Note on generic domains

- `PIN` is used by many non-TA ribonucleases; we count it only under family VapBC and flag it as "broad" in the coverage audit.
- Some antitoxins share `HTH_24`, `HTH_37` — we don't include generic HTH names in the panel to avoid contamination.


In [1]:
spark = get_spark_session()

import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path('../data')
DATA.mkdir(exist_ok=True)


## 1. Load the TA family panel


In [2]:
panel = pd.read_csv(DATA / 'ta_families_seed.tsv', sep='\t')
print(f"Loaded {len(panel)} TA families")
panel


Loaded 10 TA families


,family,toxin_pfam_name,antitoxin_pfam_name,notes
0,RelBE,RelE,RelB,Ribonuclease; ribosome-dependent mRNA cleavage
1,MazEF,PemK_toxin,MazE_antitoxin,Endoribonuclease; ACA-site cleavage. PemK_toxi...
2,ParDE,ParE_toxin,ParD_antitoxin,Gyrase inhibitor; plasmid maintenance
3,YoeB-YefM,RelE,PhdYeFM_antitox,YoeB uses the RelE-family toxin domain; antito...
4,CcdAB,CcdB,CcdA,F-plasmid classic; gyrase poison
5,HipBA,HipA_C,HigA,Serine/threonine kinase; persister formation. ...
6,VapBC,PIN,VapB_antitoxin,VapC is PIN-domain nuclease. PIN is used by MA...
7,HicAB,HicA,HicB,Ribonuclease; HicA Pfam name is 'HicA' in TADB...
8,HigBA,HigB_toxin,HigA,Stress response; ribonuclease
9,Zeta-Epsilon,Zeta_toxin,VapB_antitoxin,Zeta phosphotransferase; epsilon antitoxin oft...


In [3]:
# De-duplicated set of Pfam NAMES across toxin + antitoxin sides
pfam_names = sorted(
    set(panel['toxin_pfam_name'].dropna())
    | set(panel['antitoxin_pfam_name'].dropna())
)
print(f"Panel contains {len(pfam_names)} unique Pfam names:")
print(', '.join(pfam_names))


Panel contains 17 unique Pfam names:
CcdA, CcdB, HicA, HicB, HigA, HigB_toxin, HipA_C, MazE_antitoxin, PIN, ParD_antitoxin, ParE_toxin, PemK_toxin, PhdYeFM_antitox, RelB, RelE, VapB_antitoxin, Zeta_toxin


## 2. Baseline table sizes


In [4]:
for t in ['eggnog_mapper_annotations', 'gene_cluster', 'genome', 'pangenome', 'gtdb_species_clade']:
    n = spark.sql(f"SELECT COUNT(*) c FROM kbase_ke_pangenome.{t}").collect()[0]['c']
    print(f"{t:>32s}  {n:>14,d} rows")


       eggnog_mapper_annotations      93,558,330 rows


                    gene_cluster     132,531,501 rows


                          genome         293,059 rows


                       pangenome          27,702 rows


              gtdb_species_clade          27,690 rows


## 3. Per-name panel-coverage audit

We probe each name with a token-safe LIKE (exact match OR leading OR trailing OR interior in the comma-delimited PFAMs string). Names hitting zero are flagged for revision — the seed panel notes suggest which.


In [5]:
def token_where(col: str, name: str) -> str:
    n = name.replace("'", "''")
    return (
        f"({col} = '{n}' OR "
        f"{col} LIKE '{n},%' OR "
        f"{col} LIKE '%,{n}' OR "
        f"{col} LIKE '%,{n},%')"
    )

rows = []
for name in pfam_names:
    q = f"""
        SELECT COUNT(*) AS c
        FROM kbase_ke_pangenome.eggnog_mapper_annotations
        WHERE {token_where('PFAMs', name)}
    """
    n = spark.sql(q).collect()[0]['c']
    rows.append({'pfam_name': name, 'n_annotations': n})

coverage_df = pd.DataFrame(rows).sort_values('n_annotations', ascending=False)
print(f"Panel Pfams with zero hits: {(coverage_df['n_annotations'] == 0).sum()}")
coverage_df


Panel Pfams with zero hits: 2


,pfam_name,n_annotations
8,PIN,99214
10,ParE_toxin,61635
12,PhdYeFM_antitox,57998
7,MazE_antitoxin,43442
11,PemK_toxin,34474
6,HipA_C,30866
13,RelB,20524
3,HicB,12207
9,ParD_antitoxin,10957
16,Zeta_toxin,10161


## 4. Extract TA-carrying gene clusters


In [6]:
# Restrict to Pfam names that actually hit (drops zero-hit names)
active_names = coverage_df.loc[coverage_df['n_annotations'] > 0, 'pfam_name'].tolist()
print(f"{len(active_names)} active Pfam names of {len(pfam_names)} in panel")

# Build a UNION of token-safe queries — one per name, small distinct sets
where_clauses = " OR ".join([token_where('PFAMs', n) for n in active_names])

hits_df = spark.sql(f"""
    SELECT query_name, PFAMs, COG_category
    FROM kbase_ke_pangenome.eggnog_mapper_annotations
    WHERE {where_clauses}
""")
n_hits = hits_df.count()
print(f"Gene clusters with at least one TA Pfam hit: {n_hits:,}")


15 active Pfam names of 17 in panel


Gene clusters with at least one TA Pfam hit: 407,921


## 5. Join core/accessory status + species assignment


In [7]:
hits_df.createOrReplaceTempView("ta_hits")

joined = spark.sql("""
    SELECT
        gc.gene_cluster_id,
        gc.gtdb_species_clade_id,
        gc.is_core,
        gc.is_singleton,
        h.PFAMs,
        h.COG_category
    FROM ta_hits h
    JOIN kbase_ke_pangenome.gene_cluster gc
        ON gc.gene_cluster_id = h.query_name
""")

hits_tidy = joined.toPandas()
print(f"Rows after join to gene_cluster: {len(hits_tidy):,}")
hits_tidy.head()


Rows after join to gene_cluster: 407,118


,gene_cluster_id,gtdb_species_clade_id,is_core,is_singleton,PFAMs,COG_category
0,JAAXUO010000001.1_41,s__JACNGX01_sp013334825--GB_GCA_013334825.1,True,False,RelB,L
1,JADYXX010000063.1_191,s__PHOS-HE28_sp016705955--GB_GCA_016705955.1,True,False,PIN,S
2,JAKBPQ010000203.1_3,s__JAKAQV01_sp021846045--GB_GCA_021846045.1,False,True,PhdYeFM_antitox,D
3,CAMDGE010000028.1_1,s__SYLG01_sp009691815--GB_GCA_009691815.1,False,True,PIN,S
4,JADFXP010000002.1_218,s__JADFXP01_sp015233465--GB_GCA_015233465.1,True,False,ParE_toxin,DJ


## 6. Explode PFAMs → (gene_cluster, matched_pfam_name, side, family)


In [8]:
name_set = set(active_names)

def matched_names(pfams_str: str) -> list[str]:
    if not isinstance(pfams_str, str) or pfams_str == '-':
        return []
    tokens = {tok.strip() for tok in pfams_str.split(',')}
    return sorted(tokens & name_set)

hits_tidy['matched_names'] = hits_tidy['PFAMs'].apply(matched_names)
hits_long = hits_tidy.explode('matched_names').rename(columns={'matched_names': 'pfam_name'})
hits_long = hits_long.dropna(subset=['pfam_name'])
print(f"Long-form rows (gene_cluster × pfam_name): {len(hits_long):,}")

# Annotate with family and toxin/antitoxin side (allow multi-family membership)
side_map: dict[str, str] = {}
family_map: dict[str, str] = {}
for _, row in panel.iterrows():
    fam = row['family']
    if pd.notna(row['toxin_pfam_name']):
        # If a name appears as both toxin and antitoxin (e.g. RelE for two families),
        # keep the toxin classification for the first-seen family
        side_map.setdefault(row['toxin_pfam_name'], 'toxin')
        family_map.setdefault(row['toxin_pfam_name'], fam)
    if pd.notna(row['antitoxin_pfam_name']):
        side_map.setdefault(row['antitoxin_pfam_name'], 'antitoxin')
        family_map.setdefault(row['antitoxin_pfam_name'], fam)

hits_long['side'] = hits_long['pfam_name'].map(side_map)
hits_long['family'] = hits_long['pfam_name'].map(family_map)
hits_long.head()


Long-form rows (gene_cluster × pfam_name): 407,883


,gene_cluster_id,gtdb_species_clade_id,is_core,is_singleton,PFAMs,COG_category,pfam_name,side,family
0,JAAXUO010000001.1_41,s__JACNGX01_sp013334825--GB_GCA_013334825.1,True,False,RelB,L,RelB,antitoxin,RelBE
1,JADYXX010000063.1_191,s__PHOS-HE28_sp016705955--GB_GCA_016705955.1,True,False,PIN,S,PIN,toxin,VapBC
2,JAKBPQ010000203.1_3,s__JAKAQV01_sp021846045--GB_GCA_021846045.1,False,True,PhdYeFM_antitox,D,PhdYeFM_antitox,antitoxin,YoeB-YefM
3,CAMDGE010000028.1_1,s__SYLG01_sp009691815--GB_GCA_009691815.1,False,True,PIN,S,PIN,toxin,VapBC
4,JADFXP010000002.1_218,s__JADFXP01_sp015233465--GB_GCA_015233465.1,True,False,ParE_toxin,DJ,ParE_toxin,toxin,ParDE


## 7. Persist gene-cluster-level TSV


In [9]:
out1 = DATA / 'ta_hits_by_gene_cluster.tsv'
hits_long[
    ['gene_cluster_id', 'gtdb_species_clade_id', 'is_core', 'is_singleton',
     'pfam_name', 'side', 'family', 'PFAMs', 'COG_category']
].to_csv(out1, sep='\t', index=False)
print(f"Wrote {out1}  ({out1.stat().st_size / 1e6:.1f} MB)")

out_cov = DATA / 'ta_panel_coverage.tsv'
coverage_df['side'] = coverage_df['pfam_name'].map(side_map)
coverage_df['family'] = coverage_df['pfam_name'].map(family_map)
coverage_df.to_csv(out_cov, sep='\t', index=False)
print(f"Wrote {out_cov}")


Wrote ../data/ta_hits_by_gene_cluster.tsv  (46.7 MB)
Wrote ../data/ta_panel_coverage.tsv


## 8. Species-level TA counts

Per-species: TA-carrying gene clusters partitioned into core / accessory / singleton buckets. A gene cluster hitting multiple TA Pfams counts once.


In [10]:
def bin_status(row):
    if row['is_core']:
        return 'core'
    if row['is_singleton']:
        return 'singleton'
    return 'accessory'

gene_cluster_status = (
    hits_long.drop_duplicates(subset=['gene_cluster_id'])
    .assign(status=lambda d: d.apply(bin_status, axis=1))
)

per_species = (
    gene_cluster_status.groupby(['gtdb_species_clade_id', 'status'])
    .size().unstack(fill_value=0)
    .reindex(columns=['core', 'accessory', 'singleton'], fill_value=0)
    .reset_index()
    .rename(columns={'core': 'ta_core', 'accessory': 'ta_accessory', 'singleton': 'ta_singleton'})
)
per_species['ta_total'] = per_species[['ta_core', 'ta_accessory', 'ta_singleton']].sum(axis=1)
print(f"Species with at least one TA hit: {len(per_species):,}")
per_species.head()


Species with at least one TA hit: 25,043


status,gtdb_species_clade_id,ta_core,ta_accessory,ta_singleton,ta_total
0,s__0-14-0-80-60-11_sp018897875--GB_GCA_0188978...,24,5,5,34
1,s__0-14-3-00-41-53_sp002780895--GB_GCA_0027808...,24,24,18,66
2,s__01-FULL-36-15b_sp001782035--GB_GCA_001782035.1,1,0,0,1
3,s__01-FULL-44-24b_sp001793235--GB_GCA_001793235.1,25,0,1,26
4,s__01-FULL-45-10b_sp001804205--GB_GCA_001804205.1,5,0,3,8


In [11]:
# Family composition per species: for each TA family, count of gene clusters carrying that family
per_species_family = (
    gene_cluster_status.groupby(['gtdb_species_clade_id', 'family'])
    .size().unstack(fill_value=0)
    .reset_index()
)
print(f"Family composition matrix: {per_species_family.shape}")
per_species_family.head()


Family composition matrix: (25043, 11)


family,gtdb_species_clade_id,CcdAB,HicAB,HigBA,HipBA,MazEF,ParDE,RelBE,VapBC,YoeB-YefM,Zeta-Epsilon
0,s__0-14-0-80-60-11_sp018897875--GB_GCA_0188978...,0,1,0,0,6,13,2,9,3,0
1,s__0-14-3-00-41-53_sp002780895--GB_GCA_0027808...,1,1,0,0,15,8,3,32,4,2
2,s__01-FULL-36-15b_sp001782035--GB_GCA_001782035.1,0,0,0,0,0,0,0,1,0,0
3,s__01-FULL-44-24b_sp001793235--GB_GCA_001793235.1,0,0,0,0,4,3,2,14,3,0
4,s__01-FULL-45-10b_sp001804205--GB_GCA_001804205.1,0,0,0,0,1,2,0,2,2,1


## 9. Median genome size per species (for per-Mb normalization in NB02)


In [12]:
species_ids = per_species['gtdb_species_clade_id'].dropna().unique().tolist()
if not species_ids:
    raise RuntimeError("No species with TA hits — abort. Check the panel-coverage audit above.")

BATCH = 500
frames = []
for i in range(0, len(species_ids), BATCH):
    batch = species_ids[i:i+BATCH]
    in_clause = "', '".join([s.replace("'", "''") for s in batch])
    frames.append(spark.sql(f"""
        SELECT g.gtdb_species_clade_id,
               PERCENTILE_APPROX(m.genome_size, 0.5) AS median_size_bp,
               COUNT(*) AS n_genomes_seen
        FROM kbase_ke_pangenome.gtdb_metadata m
        JOIN kbase_ke_pangenome.genome g ON m.accession = g.genome_id
        WHERE g.gtdb_species_clade_id IN ('{in_clause}')
          AND m.genome_size IS NOT NULL
        GROUP BY g.gtdb_species_clade_id
    """).toPandas())

genome_size = pd.concat(frames, ignore_index=True)
genome_size['median_size_mb'] = genome_size['median_size_bp'] / 1e6
print(f"Species with genome-size info: {len(genome_size):,}")
genome_size.head()


Species with genome-size info: 25,037


,gtdb_species_clade_id,median_size_bp,n_genomes_seen,median_size_mb
0,s__0-14-0-80-60-11_sp018897875--GB_GCA_0188978...,4009362.0,3,4.009362
1,s__0-14-3-00-41-53_sp002780895--GB_GCA_0027808...,1986433.0,7,1.986433
2,s__01-FULL-36-15b_sp001782035--GB_GCA_001782035.1,725899.0,2,0.725899
3,s__01-FULL-44-24b_sp001793235--GB_GCA_001793235.1,898253.0,2,0.898253
4,s__01-FULL-45-10b_sp001804205--GB_GCA_001804205.1,1790555.0,2,1.790555


## 10. Persist per-species outputs


In [13]:
out2 = DATA / 'ta_per_species.tsv'
merged = per_species.merge(
    genome_size[['gtdb_species_clade_id', 'median_size_mb', 'n_genomes_seen']],
    on='gtdb_species_clade_id', how='left')
merged['ta_per_mb'] = merged['ta_total'] / merged['median_size_mb']
merged.to_csv(out2, sep='\t', index=False)
print(f"Wrote {out2}  ({merged.shape[0]:,} species)")
merged.head()


Wrote ../data/ta_per_species.tsv  (25,043 species)


,gtdb_species_clade_id,ta_core,ta_accessory,ta_singleton,ta_total,median_size_mb,n_genomes_seen,ta_per_mb
0,s__0-14-0-80-60-11_sp018897875--GB_GCA_0188978...,24,5,5,34,4.009362,3.0,8.480152
1,s__0-14-3-00-41-53_sp002780895--GB_GCA_0027808...,24,24,18,66,1.986433,7.0,33.225384
2,s__01-FULL-36-15b_sp001782035--GB_GCA_001782035.1,1,0,0,1,0.725899,2.0,1.377602
3,s__01-FULL-44-24b_sp001793235--GB_GCA_001793235.1,25,0,1,26,0.898253,2.0,28.945074
4,s__01-FULL-45-10b_sp001804205--GB_GCA_001804205.1,5,0,3,8,1.790555,2.0,4.467888


In [14]:
out3 = DATA / 'ta_family_composition_per_species.tsv'
per_species_family.to_csv(out3, sep='\t', index=False)
print(f"Wrote {out3}  ({per_species_family.shape[0]:,} species × {per_species_family.shape[1]-1} families)")


Wrote ../data/ta_family_composition_per_species.tsv  (25,043 species × 10 families)


## 11. Species-wide gene-cluster baseline

Genome-wide core/accessory/singleton counts per species (all gene clusters, not just TA). NB02 uses these as the null baseline for H1.


In [15]:
frames = []
for i in range(0, len(species_ids), BATCH):
    batch = species_ids[i:i+BATCH]
    in_clause = "', '".join([s.replace("'", "''") for s in batch])
    frames.append(spark.sql(f"""
        SELECT
            gtdb_species_clade_id,
            SUM(CASE WHEN is_core THEN 1 ELSE 0 END) AS all_core,
            SUM(CASE WHEN is_singleton THEN 1 ELSE 0 END) AS all_singleton,
            SUM(CASE WHEN NOT is_core AND NOT is_singleton THEN 1 ELSE 0 END) AS all_accessory,
            COUNT(*) AS all_gene_clusters
        FROM kbase_ke_pangenome.gene_cluster
        WHERE gtdb_species_clade_id IN ('{in_clause}')
        GROUP BY gtdb_species_clade_id
    """).toPandas())

baseline = pd.concat(frames, ignore_index=True)
out5 = DATA / 'species_gene_cluster_baseline.tsv'
baseline.to_csv(out5, sep='\t', index=False)
print(f"Wrote {out5}  ({len(baseline):,} species)")
baseline.head()


Wrote ../data/species_gene_cluster_baseline.tsv  (25,043 species)


,gtdb_species_clade_id,all_core,all_singleton,all_accessory,all_gene_clusters
0,s__AAA044-D11_sp002367715--GB_GCA_002367715.1,815,893,0,1708
1,s__AK92_sp003399765--RS_GCF_003399765.1,4131,1631,0,5762
2,s__14-2_sp011960065--GB_GCA_910575555.1,2967,3361,2861,9189
3,s__Acetatifactor_sp910585615--GB_GCA_910585615.1,2600,1992,1154,5746
4,s__AV133_sp003219265--GB_GCA_003219265.1,959,2328,872,4159


## 12. Summary

Outputs:

| File | Purpose |
|---|---|
| `ta_hits_by_gene_cluster.tsv` | Long-form: one row per gene-cluster × ta_pfam_name hit |
| `ta_per_species.tsv` | Per-species TA counts (core / accessory / singleton) + genome size + per-Mb rate |
| `ta_family_composition_per_species.tsv` | Species × families matrix for H3 |
| `ta_panel_coverage.tsv` | Panel audit — flags zero-hit names |
| `species_gene_cluster_baseline.tsv` | Per-species genome-wide core/accessory baseline for H1 |

NB02 tests H1 (paired Wilcoxon + pooled chi-square) and H2 (Mann-Whitney U + rank-biserial on ta_per_mb by lifestyle) locally. NB03 will address H3 (family composition) + phylum stratification.
